# Part C: Controlled Ablations & Tuning — Logistic Regression (Champion)

This notebook performs **4 controlled experiments** on the champion model (Logistic Regression) selected in the all-in-one pipeline notebook.

**Prerequisites:**
- `logistic_regression_baseline.joblib` — saved champion artifact (pipeline + threshold + label_encoder)
- `X_train.csv`, `y_train.csv`, `X_test.csv`, `y_test.csv` — prepared data
- `medical_weights.csv` — domain-expert feature tier/weight mapping

**Medical Evidence Tiers:**

| Tier | Weight | Evidence Level |
|------|--------|---------------|
| T1 | 4.0 | Strongest clinical association with heart disease |
| T2 | 2.0 | Strong secondary risk factors |
| T3 | 1.0 | Moderate/indirect association |
| T4 | 0.5 | Weak association |
| T5 | 0.0 | Little to no clinical evidence |

**Ablation Log:**

| # | Hypothesis | Controlled Change | IV | DV |
|---|-----------|-------------------|----|----|
| 1 | Composite medical risk score improves detection | Add `MedicalRiskScore` feature (tier-weighted sum) | Presence of risk score feature | CV F1(Yes) mean ± std |
| 2 | Removing low-evidence features reduces noise | Drop tiers progressively (T5 → T4 → T3) + combinations | Feature set composition | CV F1(Yes) mean ± std |
| 3 | More aggressive class weight improves recall/F1 | Fix class_weight to {0:1, 1:K} for K > 5 | Positive-class weight K | CV F1(Yes) mean ± std |
| 4 | F2-optimal threshold better suits medical screening | Select threshold by F2-score (recall-biased) | Threshold selection criterion | Test F1, F2, Recall, Precision |

## 1. Setup: Load Artifacts & Data

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
import warnings
warnings.filterwarnings("ignore")

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import (
    classification_report,
    f1_score,
    precision_score,
    recall_score,
    accuracy_score,
    roc_auc_score,
    brier_score_loss,
    log_loss,
    fbeta_score,
    precision_recall_curve,
    ConfusionMatrixDisplay,
)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Libraries imported.")

Libraries imported.


In [6]:
# Load champion model artifact
artifact = joblib.load("logistic_regression_baseline.joblib")
baseline_pipeline = artifact["pipeline"]
baseline_threshold = artifact["threshold"]
le = artifact["label_encoder"]

print(f"Loaded champion: Logistic Regression")
print(f"Baseline threshold: {baseline_threshold}")
print(f"Pipeline: {baseline_pipeline}")

# Extract baseline hyperparameters for reuse in ablations
baseline_params = baseline_pipeline.named_steps["logreg"].get_params()
print(f"\nBaseline hyperparameters:")
print(f"  C:            {baseline_params['C']}")
print(f"  class_weight: {baseline_params['class_weight']}")
print(f"  solver:       {baseline_params['solver']}")
print(f"  penalty:      {baseline_params['penalty']}")

Loaded champion: Logistic Regression
Baseline threshold: 0.5
Pipeline: Pipeline(steps=[('scaler', StandardScaler()),
                ('logreg',
                 LogisticRegression(C=np.float64(0.02069138081114789),
                                    class_weight={0: 1, 1: 5}, max_iter=5000,
                                    penalty='l1', random_state=42,
                                    solver='liblinear'))])

Baseline hyperparameters:
  C:            0.02069138081114789
  class_weight: {0: 1, 1: 5}
  solver:       liblinear
  penalty:      l1


In [7]:
# Load prepared data
X_train = pd.read_csv("X_train.csv")
X_test = pd.read_csv("X_test.csv")
y_train_raw = pd.read_csv("y_train.csv").values.ravel()
y_test_raw = pd.read_csv("y_test.csv").values.ravel()

y_train = le.transform(y_train_raw)
y_test = le.transform(y_test_raw)

print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")
print(f"Positive rate (train): {np.mean(y_train):.4f}")
print(f"Positive rate (test):  {np.mean(y_test):.4f}")

X_train: (352827, 76)
X_test:  (88414, 76)
Positive rate (train): 0.0568
Positive rate (test):  0.0568


In [8]:
# Load medical weights CSV
# Format: column headers = feature names, row 0 = tier (T1-T5), row 1 = weight (4, 2, 1, 0.5, 0)
medical_weights = pd.read_csv("medical_weights.csv")

# Parse into dictionaries
weight_map = {}  # feature_name -> weight
tier_map = {}    # feature_name -> tier

for col in medical_weights.columns:
    feature_name = col.strip()
    tier = str(medical_weights[col].iloc[0]).strip()
    weight = float(medical_weights[col].iloc[1])
    weight_map[feature_name] = weight
    tier_map[feature_name] = tier

# Group features by tier
tier_features = {}
for feat, tier in tier_map.items():
    tier_features.setdefault(tier, []).append(feat)

print("Medical weights loaded:")
print("=" * 60)
for tier in sorted(tier_features.keys()):
    features = tier_features[tier]
    w = weight_map[features[0]]
    print(f"\n  {tier} (weight={w}):")
    for feat in features:
        print(f"    - {feat}")

print(f"\nTotal features mapped: {len(weight_map)}")

Medical weights loaded:

  T1 (weight=4.0):
    - HadHeartAttack
    - HadStroke
    - HadCOPD
    - HadDiabetes_No
    - HadDiabetes_No, pre-diabetes or borderline diabetes
    - HadDiabetes_Yes
    - HadDiabetes_Yes, but only during pregnancy (female)
    - SmokerStatus_Current smoker - now smokes every day
    - SmokerStatus_Current smoker - now smokes some days
    - SmokerStatus_Former smoker
    - SmokerStatus_Never smoked
    - ECigaretteUsage_Never used e-cigarettes in my entire life
    - ECigaretteUsage_Not at all (right now)
    - ECigaretteUsage_Use them every day
    - ECigaretteUsage_Use them some days
    - BMI

  T2 (weight=2.0):
    - GeneralHealth_Excellent
    - GeneralHealth_Very good
    - GeneralHealth_Good
    - GeneralHealth_Fair
    - GeneralHealth_Poor
    - PhysicalHealthDays
    - PhysicalActivities
    - HadAsthma
    - HadKidneyDisease
    - HadArthritis
    - DifficultyWalking
    - AgeCategory_Age 18 to 24
    - AgeCategory_Age 25 to 29
    - AgeCategory

## 2. Shared Evaluation Helpers

In [9]:
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

TARGET_NAMES = ["No Heart Disease (0)", "Heart Disease (1)"]


def evaluate_model(pipeline, X_tr, y_tr, X_te, y_te, threshold, model_label="Model"):
    """Run 5-fold CV on training data and evaluate on test set at given threshold."""
    cv_scores = cross_val_score(pipeline, X_tr, y_tr, cv=cv_strategy, scoring="f1", n_jobs=-1)
    cv_mean, cv_std = cv_scores.mean(), cv_scores.std()
    
    pipeline.fit(X_tr, y_tr)
    y_proba = pipeline.predict_proba(X_te)[:, 1]
    y_pred = (y_proba >= threshold).astype(int)
    
    metrics = {
        "Label": model_label,
        "CV F1 (mean)": cv_mean,
        "CV F1 (std)": cv_std,
        "CV F1": f"{cv_mean:.4f} +/- {cv_std:.4f}",
        "Threshold": threshold,
        "Test Accuracy": accuracy_score(y_te, y_pred),
        "Test F1 (Yes)": f1_score(y_te, y_pred),
        "Test Precision (Yes)": precision_score(y_te, y_pred, zero_division=0),
        "Test Recall (Yes)": recall_score(y_te, y_pred, zero_division=0),
        "Test ROC-AUC": roc_auc_score(y_te, y_proba),
        "Test Brier": brier_score_loss(y_te, y_proba),
        "Test Log Loss": log_loss(y_te, y_proba),
    }
    
    return metrics, y_proba, y_pred


def build_baseline_pipeline():
    """Reconstruct the baseline LR pipeline with the same hyperparameters."""
    return Pipeline([
        ("scaler", StandardScaler()),
        ("logreg", LogisticRegression(
            C=baseline_params["C"],
            class_weight=baseline_params["class_weight"],
            solver=baseline_params["solver"],
            penalty=baseline_params["penalty"],
            max_iter=5000,
            random_state=RANDOM_STATE,
        )),
    ])


def print_ablation_result(baseline_metrics, ablation_metrics, ablation_name):
    """Print a side-by-side comparison of baseline vs ablation."""
    print(f"\n{'=' * 80}")
    print(f"  {ablation_name}")
    print(f"{'=' * 80}")
    
    compare_keys = ["CV F1", "Threshold", "Test F1 (Yes)", "Test Precision (Yes)",
                    "Test Recall (Yes)", "Test ROC-AUC", "Test Brier"]
    
    print(f"\n{'Metric':<25} {'Baseline':<25} {'Ablation':<25} {'Delta':<15}")
    print("-" * 80)
    for key in compare_keys:
        bval = baseline_metrics[key]
        aval = ablation_metrics[key]
        if isinstance(bval, float):
            delta = aval - bval
            if key == "Test Brier":
                direction = "(better)" if delta < 0 else "(worse)"
            else:
                direction = "(better)" if delta > 0 else "(worse)" if delta < 0 else ""
            print(f"  {key:<23} {bval:<25.4f} {aval:<25.4f} {delta:+.4f} {direction}")
        else:
            print(f"  {key:<23} {str(bval):<25} {str(aval):<25}")


def get_columns_for_features(X, feature_names):
    """Find all columns in X that match the given feature names (direct or OHE prefix)."""
    cols = []
    for feat in feature_names:
        if feat in X.columns:
            cols.append(feat)
        else:
            ohe_cols = [c for c in X.columns if c.startswith(feat + "_")]
            cols.extend(ohe_cols)
    return cols


print("Helpers defined.")

Helpers defined.


## 3. Baseline Measurement (Control)

In [10]:
baseline_pipe = build_baseline_pipeline()
baseline_metrics, baseline_proba, baseline_pred = evaluate_model(
    baseline_pipe, X_train, y_train, X_test, y_test,
    threshold=baseline_threshold,
    model_label="Baseline"
)

print("Baseline Champion (Logistic Regression) Metrics:")
print("=" * 60)
for k, v in baseline_metrics.items():
    if k == "Label":
        continue
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

Baseline Champion (Logistic Regression) Metrics:
  CV F1 (mean): 0.3250
  CV F1 (std): 0.0035
  CV F1: 0.3250 +/- 0.0035
  Threshold: 0.5000
  Test Accuracy: 0.9004
  Test F1 (Yes): 0.3166
  Test Precision (Yes): 0.2593
  Test Recall (Yes): 0.4062
  Test ROC-AUC: 0.8389
  Test Brier: 0.0763
  Test Log Loss: 0.2611


---
## Ablation 1: Medical Tier-Weighted Risk Score Feature

**Hypothesis:** Adding a composite `MedicalRiskScore` feature — a weighted sum of existing features using tier weights derived from peer-reviewed cardiology literature — will improve heart disease detection by encoding domain knowledge that raw features alone do not capture.

**How it works:** Each feature column is multiplied by its tier weight (T1=4, T2=2, T3=1, T4=0.5, T5=0) and summed into a single score. For one-hot encoded features (e.g., `SmokerStatus`), all dummy columns share the same tier weight. The weight tells the model "pay more attention to this group of features" — but the model still learns *how* each individual value (e.g., "smokes every day" vs "never smoked") contributes to risk through its own coefficients.

**Controlled Change:** Add one new feature (`MedicalRiskScore`). All pipeline components remain identical.

**IV:** Presence/absence of `MedicalRiskScore`.
**DV:** 5-fold stratified CV F1(Yes) mean ± std.

In [11]:
def compute_risk_score(X, weight_map, column_list):
    """
    Compute a weighted risk score from the feature matrix.
    
    For each key in weight_map:
    - If it matches a column directly, multiply by weight
    - If it matches as a prefix (one-hot encoded), multiply all matching dummy columns by weight
    
    Note: T5 features have weight=0 so they contribute nothing to the score.
    This is intentional — they are "no evidence" features.
    """
    score = np.zeros(len(X))
    matched, unmatched = [], []
    
    for feature, weight in weight_map.items():
        if weight == 0:
            matched.append((feature, "T5 (weight=0, skipped)", weight))
            continue
            
        if feature in column_list:
            score += X[feature].values * weight
            matched.append((feature, "direct", weight))
        else:
            ohe_cols = [c for c in column_list if c.startswith(feature + "_")]
            if ohe_cols:
                for ohe_col in ohe_cols:
                    score += X[ohe_col].values * weight
                matched.append((feature, f"{len(ohe_cols)} OHE cols", weight))
            else:
                unmatched.append(feature)
    
    return score, matched, unmatched


risk_train, matched, unmatched = compute_risk_score(X_train, weight_map, list(X_train.columns))

print("Feature matching for MedicalRiskScore:")
print("-" * 60)
for feat, match_type, w in matched:
    print(f"  {feat:<40} ({match_type}, weight={w})")

if unmatched:
    print(f"\nWARNING — Unmatched features ({len(unmatched)}):")
    for feat in unmatched:
        print(f"  {feat}")
else:
    print("\nAll features matched successfully.")

Feature matching for MedicalRiskScore:
------------------------------------------------------------
  Sex                                      (direct, weight=1.0)
  GeneralHealth_Excellent                  (direct, weight=2.0)
  GeneralHealth_Very good                  (direct, weight=2.0)
  GeneralHealth_Good                       (direct, weight=2.0)
  GeneralHealth_Fair                       (direct, weight=2.0)
  GeneralHealth_Poor                       (direct, weight=2.0)
  PhysicalHealthDays                       (direct, weight=2.0)
  MentalHealthDays                         (direct, weight=1.0)
  LastCheckupTime_5 or more years ago      (T5 (weight=0, skipped), weight=0.0)
  LastCheckupTime_Within past year (anytime less than 12 months ago) (T5 (weight=0, skipped), weight=0.0)
  LastCheckupTime_Within past 2 years (1 year but less than 2 years ago) (T5 (weight=0, skipped), weight=0.0)
  LastCheckupTime_Within past 5 years (2 years but less than 5 years ago) (T5 (weight=0, ski

In [12]:
# Create augmented datasets
X_train_abl1 = X_train.copy()
X_test_abl1 = X_test.copy()

risk_train, _, _ = compute_risk_score(X_train, weight_map, list(X_train.columns))
risk_test, _, _ = compute_risk_score(X_test, weight_map, list(X_test.columns))

X_train_abl1["MedicalRiskScore"] = risk_train
X_test_abl1["MedicalRiskScore"] = risk_test

print(f"Added MedicalRiskScore feature.")
print(f"  Train: range=[{risk_train.min():.2f}, {risk_train.max():.2f}], mean={risk_train.mean():.2f}, std={risk_train.std():.2f}")
print(f"  Test:  range=[{risk_test.min():.2f}, {risk_test.max():.2f}], mean={risk_test.mean():.2f}")
print(f"  New shape: {X_train_abl1.shape}")

# Evaluate
abl1_pipe = build_baseline_pipeline()
abl1_metrics, abl1_proba, abl1_pred = evaluate_model(
    abl1_pipe, X_train_abl1, y_train, X_test_abl1, y_test,
    threshold=baseline_threshold,
    model_label="Ablation 1: +MedicalRiskScore"
)

print_ablation_result(baseline_metrics, abl1_metrics, "ABLATION 1: Medical Risk Score Feature")

Added MedicalRiskScore feature.
  Train: range=[73.94, 370.18], mean=156.91, std=34.40
  Test:  range=[73.74, 376.54], mean=156.85
  New shape: (352827, 77)

  ABLATION 1: Medical Risk Score Feature

Metric                    Baseline                  Ablation                  Delta          
--------------------------------------------------------------------------------
  CV F1                   0.3250 +/- 0.0035         0.3251 +/- 0.0036        
  Threshold               0.5000                    0.5000                    +0.0000 
  Test F1 (Yes)           0.3166                    0.3166                    +0.0000 (better)
  Test Precision (Yes)    0.2593                    0.2594                    +0.0000 (better)
  Test Recall (Yes)       0.4062                    0.4062                    +0.0000 
  Test ROC-AUC            0.8389                    0.8389                    -0.0000 (worse)
  Test Brier              0.0763                    0.0763                    -0.0000 (be

### Ablation 1: Conclusion

*Fill in after running:*
- Did CV F1 improve, degrade, or remain stable?
- Did the risk score provide additive signal beyond the raw features?
- Note: LR already has access to all the individual features and can learn its own linear combination of them. The risk score may be partially redundant — but it could still help by providing a pre-computed "summary signal" that makes the optimisation landscape easier to navigate.

---
## Ablation 2: Medical Tier-Based Feature Selection (Noise Removal)

**Hypothesis:** Removing features with the weakest evidence-based association with heart disease will improve generalisation by reducing noise, without losing clinically meaningful signal.

**Controlled Change:** Progressively drop features by tier, then test combinations:
- **Phase A:** Drop T5 only
- **Phase B:** Drop T5 + T4
- **Phase C:** Drop T5 + T4 + T3
- **Phase D:** Drop T5 + T3 (skip T4, keep T4)
- **Phase E:** Drop T5 + T4 + T3 combined (same as C, but explicitly labelled for the log)
- **Phase F:** Drop T4 + T3 only (keep T5 — tests if T5 features carry unexpected signal)

This progressive approach lets us see the incremental effect of each tier removal and identify which tiers are noise vs. which carry hidden signal despite weak medical evidence.

**IV:** Feature set composition.
**DV:** 5-fold stratified CV F1(Yes) mean ± std.

In [13]:
# Display what will be dropped in each phase
print("Feature tiers to be tested:")
print("=" * 60)
for tier in sorted(tier_features.keys()):
    cols = get_columns_for_features(X_train, tier_features.get(tier, []))
    print(f"\n  {tier}: {tier_features.get(tier, [])})")
    print(f"    -> {len(cols)} columns in X_train: {cols[:5]}{'...' if len(cols) > 5 else ''}")

Feature tiers to be tested:

  T1: ['HadHeartAttack', 'HadStroke', 'HadCOPD', 'HadDiabetes_No', 'HadDiabetes_No, pre-diabetes or borderline diabetes', 'HadDiabetes_Yes', 'HadDiabetes_Yes, but only during pregnancy (female)', 'SmokerStatus_Current smoker - now smokes every day', 'SmokerStatus_Current smoker - now smokes some days', 'SmokerStatus_Former smoker', 'SmokerStatus_Never smoked', 'ECigaretteUsage_Never used e-cigarettes in my entire life', 'ECigaretteUsage_Not at all (right now)', 'ECigaretteUsage_Use them every day', 'ECigaretteUsage_Use them some days', 'BMI'])
    -> 15 columns in X_train: ['HadStroke', 'HadCOPD', 'HadDiabetes_No', 'HadDiabetes_No, pre-diabetes or borderline diabetes', 'HadDiabetes_Yes']...

  T2: ['GeneralHealth_Excellent', 'GeneralHealth_Very good', 'GeneralHealth_Good', 'GeneralHealth_Fair', 'GeneralHealth_Poor', 'PhysicalHealthDays', 'PhysicalActivities', 'HadAsthma', 'HadKidneyDisease', 'HadArthritis', 'DifficultyWalking', 'AgeCategory_Age 18 to 24', '

In [14]:
# Define all phases
phases = {
    "Phase A: Drop T5": ["T5"],
    "Phase B: Drop T5+T4": ["T5", "T4"],
    "Phase C: Drop T5+T4+T3": ["T5", "T4", "T3"],
    "Phase D: Drop T5+T3 (keep T4)": ["T5", "T3"],
    "Phase E: Drop T4+T3 (keep T5)": ["T4", "T3"],
}

abl2_results = {}

for phase_name, tiers_to_drop in phases.items():
    # Collect features from all tiers to drop
    features_to_drop = []
    for tier in tiers_to_drop:
        features_to_drop.extend(tier_features.get(tier, []))
    
    cols_to_drop = get_columns_for_features(X_train, features_to_drop)
    
    X_tr = X_train.drop(columns=cols_to_drop, errors="ignore")
    X_te = X_test.drop(columns=cols_to_drop, errors="ignore")
    
    print(f"\n{phase_name}")
    print(f"  Tiers dropped: {tiers_to_drop}")
    print(f"  Features dropped: {features_to_drop}")
    print(f"  Columns removed: {len(cols_to_drop)}")
    print(f"  Remaining shape: {X_tr.shape} (was {X_train.shape})")
    
    pipe = build_baseline_pipeline()
    metrics, proba, pred = evaluate_model(
        pipe, X_tr, y_train, X_te, y_test,
        threshold=baseline_threshold,
        model_label=phase_name,
    )
    
    abl2_results[phase_name] = metrics
    print_ablation_result(baseline_metrics, metrics, phase_name)


Phase A: Drop T5
  Tiers dropped: ['T5']
  Features dropped: ['LastCheckupTime_5 or more years ago', 'LastCheckupTime_Within past year (anytime less than 12 months ago)', 'LastCheckupTime_Within past 2 years (1 year but less than 2 years ago)', 'LastCheckupTime_Within past 5 years (2 years but less than 5 years ago)', 'RemovedTeeth_All', 'RemovedTeeth_6 or more, but not all', 'RemovedTeeth_1 to 5', 'RemovedTeeth_None of them', 'DeafOrHardOfHearing', 'BlindOrVisionDifficulty', 'DifficultyConcentrating', 'DifficultyDressingBathing', 'DifficultyErrands', 'ChestScan', 'HIVTesting', 'CovidPos_No', 'CovidPos_Tested positive using home test without a health professional', 'CovidPos_Yes']
  Columns removed: 18
  Remaining shape: (352827, 58) (was (352827, 76))

  Phase A: Drop T5

Metric                    Baseline                  Ablation                  Delta          
--------------------------------------------------------------------------------
  CV F1                   0.3250 +/- 0.0

In [15]:
# Summary table for all Ablation 2 phases
print("\n" + "=" * 100)
print("  ABLATION 2 SUMMARY: Progressive Feature Selection")
print("=" * 100)

abl2_rows = [{"Variant": "Baseline (all features)"} | {k: v for k, v in baseline_metrics.items() if k != "Label"}]
for phase_name, metrics in abl2_results.items():
    abl2_rows.append({"Variant": phase_name} | {k: v for k, v in metrics.items() if k != "Label"})

abl2_summary = pd.DataFrame(abl2_rows).set_index("Variant")
print(abl2_summary[["CV F1", "Test F1 (Yes)", "Test Precision (Yes)",
                      "Test Recall (Yes)", "Test ROC-AUC", "Test Brier"]].to_string())


  ABLATION 2 SUMMARY: Progressive Feature Selection
                                           CV F1  Test F1 (Yes)  Test Precision (Yes)  Test Recall (Yes)  Test ROC-AUC  Test Brier
Variant                                                                                                                           
Baseline (all features)        0.3250 +/- 0.0035       0.316574              0.259344           0.406213      0.838906    0.076251
Phase A: Drop T5               0.3190 +/- 0.0059       0.309459              0.259284           0.383712      0.832379    0.076467
Phase B: Drop T5+T4            0.3174 +/- 0.0066       0.307963              0.257460           0.383114      0.832229    0.076508
Phase C: Drop T5+T4+T3         0.3021 +/- 0.0033       0.289230              0.247107           0.348666      0.819164    0.077854
Phase D: Drop T5+T3 (keep T4)  0.3031 +/- 0.0038       0.292663              0.249719           0.353445      0.819533    0.077775
Phase E: Drop T4+T3 (keep T5) 

### Ablation 2: Conclusion

*Fill in after running:*
- Did removing T5 (no-evidence features) help?
- At which phase did performance start to degrade?
- Did any "skip" combination (e.g., Phase D: keep T4 but drop T3) reveal that T4 features carry more signal than T3?
- Which phase gives the best trade-off? This tells you the optimal feature set.

---
## Ablation 3: Tier-Informed Class Weight Tuning

**Hypothesis:** The baseline selected `class_weight = {0: 1, 1: 5}`. A more aggressive positive-class weight will improve Recall(Yes) and F1(Yes) by forcing the model to penalise missed heart disease cases more heavily. This is informed by the medical context where false negatives carry life-threatening consequences.

**Controlled Change:** Fix `class_weight` to `{0: 1, 1: K}` for K in {5, 8, 10, 12, 15, 20}. K=5 is the baseline included for reference. All other hyperparameters (C, solver, penalty) and the feature set remain identical.

**IV:** Positive-class weight multiplier K.
**DV:** 5-fold stratified CV F1(Yes) mean ± std.

In [16]:
weight_values = [5, 8, 10, 12, 15, 20]  # 5 = baseline for reference

abl3_results = []

for k in weight_values:
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("logreg", LogisticRegression(
            C=baseline_params["C"],
            class_weight={0: 1, 1: k},
            solver=baseline_params["solver"],
            penalty=baseline_params["penalty"],
            max_iter=5000,
            random_state=RANDOM_STATE,
        )),
    ])
    
    metrics, proba, pred = evaluate_model(
        pipe, X_train, y_train, X_test, y_test,
        threshold=baseline_threshold,
        model_label=f"class_weight={{0:1, 1:{k}}}"
    )
    metrics["K"] = k
    abl3_results.append(metrics)

abl3_df = pd.DataFrame(abl3_results)

print("=" * 80)
print("  ABLATION 3: Class Weight Sweep")
print("=" * 80)
print()
print(abl3_df[["K", "CV F1", "Test F1 (Yes)", "Test Precision (Yes)",
               "Test Recall (Yes)", "Test ROC-AUC", "Test Brier"]].to_string(index=False))

  ABLATION 3: Class Weight Sweep

 K             CV F1  Test F1 (Yes)  Test Precision (Yes)  Test Recall (Yes)  Test ROC-AUC  Test Brier
 5 0.3250 +/- 0.0035       0.316574              0.259344           0.406213      0.838906    0.076251
 8 0.3150 +/- 0.0039       0.309574              0.213981           0.559538      0.838953    0.102746
10 0.3010 +/- 0.0039       0.295693              0.193292           0.628833      0.838966    0.119254
12 0.2894 +/- 0.0022       0.284669              0.179875           0.681999      0.838975    0.134697
15 0.2730 +/- 0.0014       0.268344              0.163714           0.743528      0.838979    0.155993
20 0.2506 +/- 0.0013       0.247763              0.146235           0.810434      0.838978    0.187244


In [17]:
# Plot the trade-off
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax1 = axes[0]
ax1.plot(abl3_df["K"], abl3_df["Test F1 (Yes)"], "o-", linewidth=2, label="F1 (Yes)")
ax1.plot(abl3_df["K"], abl3_df["Test Precision (Yes)"], "s--", linewidth=2, label="Precision (Yes)")
ax1.plot(abl3_df["K"], abl3_df["Test Recall (Yes)"], "^--", linewidth=2, label="Recall (Yes)")
ax1.axvline(5, color="grey", linestyle=":", alpha=0.7, label="Baseline (K=5)")
ax1.set_xlabel("Positive Class Weight (K)")
ax1.set_ylabel("Score")
ax1.set_title("Ablation 3: F1 / Precision / Recall vs Class Weight")
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2 = axes[1]
ax2.plot(abl3_df["K"], abl3_df["CV F1 (mean)"], "o-", color="purple", linewidth=2)
ax2.fill_between(abl3_df["K"],
                  abl3_df["CV F1 (mean)"] - abl3_df["CV F1 (std)"],
                  abl3_df["CV F1 (mean)"] + abl3_df["CV F1 (std)"],
                  alpha=0.2, color="purple")
ax2.axvline(5, color="grey", linestyle=":", alpha=0.7, label="Baseline (K=5)")
ax2.set_xlabel("Positive Class Weight (K)")
ax2.set_ylabel("CV F1 (mean +/- std)")
ax2.set_title("Ablation 3: Cross-Validation Stability")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("ablation3_class_weight.png", dpi=150, bbox_inches="tight")
plt.close()
print("Plot saved: ablation3_class_weight.png")

Plot saved: ablation3_class_weight.png


In [18]:
# Identify best K
best_abl3 = abl3_df.loc[abl3_df["Test F1 (Yes)"].idxmax()]
print(f"Best class weight: K={int(best_abl3['K'])}")
print(f"  Test F1 (Yes): {best_abl3['Test F1 (Yes)']:.4f} (baseline: {baseline_metrics['Test F1 (Yes)']:.4f})")
print(f"  Test Recall:   {best_abl3['Test Recall (Yes)']:.4f} (baseline: {baseline_metrics['Test Recall (Yes)']:.4f})")
print(f"  Test Precision: {best_abl3['Test Precision (Yes)']:.4f} (baseline: {baseline_metrics['Test Precision (Yes)']:.4f})")

Best class weight: K=5
  Test F1 (Yes): 0.3166 (baseline: 0.3166)
  Test Recall:   0.4062 (baseline: 0.4062)
  Test Precision: 0.2593 (baseline: 0.2593)


### Ablation 3: Conclusion

*Fill in after running:*
- Did increasing K beyond 5 improve F1(Yes)?
- At what K does recall plateau while precision collapses?
- Is there a "sweet spot" that improves recall meaningfully without destroying precision?
- Note the CV stability (std) — a large std at high K suggests the model becomes unstable.

---
## Ablation 4: Threshold Re-Optimisation (F2-Score for Medical Screening)

**Hypothesis:** The baseline threshold of 0.50 was selected by maximising F1, which weights precision and recall equally. In a medical screening context, a false negative (missed heart disease) carries greater risk than a false positive (unnecessary follow-up). Selecting the threshold by F2-score — which weights recall 2x more than precision — will yield a better operating point for clinical triage.

**Justification:** Population health screening programmes favour sensitivity over specificity. The cost of a missed diagnosis (delayed treatment, potential fatality) far outweighs a false alarm (additional diagnostic test). The F2-score operationalises this asymmetry: F_beta = (1+beta^2) * (P*R) / (beta^2*P + R), with beta=2.

**Controlled Change:** Re-sweep thresholds 0.10–0.90 and select by max F2-score instead of F1. The model (pipeline + hyperparameters) remains identical.

**IV:** Threshold selection criterion (F1-optimal vs F2-optimal).
**DV:** Test F1, F2, Recall, Precision.

In [19]:
# Use the baseline pipeline fitted fresh
baseline_pipe_fitted = build_baseline_pipeline()
baseline_pipe_fitted.fit(X_train, y_train)
y_proba_abl4 = baseline_pipe_fitted.predict_proba(X_test)[:, 1]

# Sweep thresholds at fine granularity
thresholds_sweep = np.arange(0.10, 0.91, 0.01)

sweep_results = []
for t in thresholds_sweep:
    y_pred_t = (y_proba_abl4 >= t).astype(int)
    sweep_results.append({
        "Threshold": round(t, 2),
        "F1": f1_score(y_test, y_pred_t, zero_division=0),
        "F2": fbeta_score(y_test, y_pred_t, beta=2, zero_division=0),
        "Precision": precision_score(y_test, y_pred_t, zero_division=0),
        "Recall": recall_score(y_test, y_pred_t, zero_division=0),
        "Accuracy": accuracy_score(y_test, y_pred_t),
    })

sweep_df = pd.DataFrame(sweep_results)

f1_best_row = sweep_df.loc[sweep_df["F1"].idxmax()]
f2_best_row = sweep_df.loc[sweep_df["F2"].idxmax()]

print("=" * 80)
print("  ABLATION 4: Threshold Selection Comparison")
print("=" * 80)
print(f"\n{'Metric':<20} {'F1-Optimal (t='}{f1_best_row['Threshold']:.2f}){')':<15} {'F2-Optimal (t='}{f2_best_row['Threshold']:.2f}){')':<15}")
print("-" * 80)
for metric in ["F1", "F2", "Precision", "Recall", "Accuracy"]:
    print(f"  {metric:<18} {f1_best_row[metric]:<30.4f} {f2_best_row[metric]:<30.4f}")

  ABLATION 4: Threshold Selection Comparison

Metric               F1-Optimal (t=0.47))               F2-Optimal (t=0.27))              
--------------------------------------------------------------------------------
  F1                 0.3190                         0.2769                        
  F2                 0.3844                         0.4386                        
  Precision          0.2486                         0.1716                        
  Recall             0.4452                         0.7178                        
  Accuracy           0.8920                         0.7871                        


In [20]:
# Plot F1 vs F2 threshold curves
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(sweep_df["Threshold"], sweep_df["F1"], "o-", markersize=3, linewidth=2, label="F1 Score", color="blue")
ax.plot(sweep_df["Threshold"], sweep_df["F2"], "s-", markersize=3, linewidth=2, label="F2 Score", color="green")
ax.plot(sweep_df["Threshold"], sweep_df["Precision"], "--", linewidth=1.5, label="Precision", color="orange", alpha=0.7)
ax.plot(sweep_df["Threshold"], sweep_df["Recall"], "--", linewidth=1.5, label="Recall", color="red", alpha=0.7)

ax.axvline(f1_best_row["Threshold"], color="blue", linestyle=":", alpha=0.6, label=f"F1-optimal (t={f1_best_row['Threshold']:.2f})")
ax.axvline(f2_best_row["Threshold"], color="green", linestyle=":", alpha=0.6, label=f"F2-optimal (t={f2_best_row['Threshold']:.2f})")

ax.set_xlabel("Decision Threshold")
ax.set_ylabel("Score")
ax.set_title("Ablation 4: F1 vs F2 Threshold Optimisation")
ax.legend(loc="best", fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("ablation4_threshold_f2.png", dpi=150, bbox_inches="tight")
plt.close()
print("Plot saved: ablation4_threshold_f2.png")

Plot saved: ablation4_threshold_f2.png


In [21]:
# Confusion matrix comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

y_pred_f1 = (y_proba_abl4 >= f1_best_row["Threshold"]).astype(int)
y_pred_f2 = (y_proba_abl4 >= f2_best_row["Threshold"]).astype(int)

ConfusionMatrixDisplay.from_predictions(y_test, y_pred_f1, display_labels=TARGET_NAMES, cmap="Blues", ax=axes[0])
axes[0].set_title(f"F1-Optimal Threshold = {f1_best_row['Threshold']:.2f}")

ConfusionMatrixDisplay.from_predictions(y_test, y_pred_f2, display_labels=TARGET_NAMES, cmap="Oranges", ax=axes[1])
axes[1].set_title(f"F2-Optimal Threshold = {f2_best_row['Threshold']:.2f}")

plt.tight_layout()
plt.savefig("ablation4_confusion_comparison.png", dpi=150, bbox_inches="tight")
plt.close()
print("Plot saved: ablation4_confusion_comparison.png")

# FN/FP analysis
fn_f1 = ((y_test == 1) & (y_pred_f1 == 0)).sum()
fn_f2 = ((y_test == 1) & (y_pred_f2 == 0)).sum()
fp_f1 = ((y_test == 0) & (y_pred_f1 == 1)).sum()
fp_f2 = ((y_test == 0) & (y_pred_f2 == 1)).sum()

print(f"\nFalse Negatives (missed heart disease cases):")
print(f"  F1-optimal: {fn_f1}")
print(f"  F2-optimal: {fn_f2}")
print(f"  Reduction:  {fn_f1 - fn_f2} fewer missed cases ({(fn_f1 - fn_f2) / fn_f1 * 100:.1f}% improvement)")

print(f"\nFalse Positives (unnecessary follow-ups):")
print(f"  F1-optimal: {fp_f1}")
print(f"  F2-optimal: {fp_f2}")
print(f"  Increase:   {fp_f2 - fp_f1} additional false alarms")

Plot saved: ablation4_confusion_comparison.png

False Negatives (missed heart disease cases):
  F1-optimal: 2786
  F2-optimal: 1417
  Reduction:  1369 fewer missed cases (49.1% improvement)

False Positives (unnecessary follow-ups):
  F1-optimal: 6759
  F2-optimal: 17407
  Increase:   10648 additional false alarms


### Ablation 4: Conclusion

*Fill in after running:*
- How much did the F2-optimal threshold shift from the F1-optimal?
- How many additional missed heart disease cases are caught?
- Is the increase in false positives acceptable for a screening context?
- This ablation does NOT change the model — it changes the business decision rule applied to the model's output.

---
## 4. Final Ablation Log (Deliverable for Part C)

In [22]:
# Find the best Ablation 2 phase
best_abl2_name = None
best_abl2_f1 = -1
for phase_name, metrics in abl2_results.items():
    if metrics["Test F1 (Yes)"] > best_abl2_f1:
        best_abl2_f1 = metrics["Test F1 (Yes)"]
        best_abl2_name = phase_name

best_abl2_metrics = abl2_results[best_abl2_name]

# Compile the formal ablation log
ablation_log = pd.DataFrame([
    {
        "Ablation": "Baseline",
        "Hypothesis": "N/A (control)",
        "Controlled Change": "None",
        "CV F1 (mean +/- std)": baseline_metrics["CV F1"],
        "Test F1 (Yes)": f"{baseline_metrics['Test F1 (Yes)']:.4f}",
        "Test Recall (Yes)": f"{baseline_metrics['Test Recall (Yes)']:.4f}",
        "Test ROC-AUC": f"{baseline_metrics['Test ROC-AUC']:.4f}",
        "Conclusion": "Control measurement",
    },
    {
        "Ablation": "1: Risk Score",
        "Hypothesis": "Domain-weighted composite feature improves detection",
        "Controlled Change": "Add MedicalRiskScore feature (T1=4, T2=2, T3=1, T4=0.5, T5=0)",
        "CV F1 (mean +/- std)": abl1_metrics["CV F1"],
        "Test F1 (Yes)": f"{abl1_metrics['Test F1 (Yes)']:.4f}",
        "Test Recall (Yes)": f"{abl1_metrics['Test Recall (Yes)']:.4f}",
        "Test ROC-AUC": f"{abl1_metrics['Test ROC-AUC']:.4f}",
        "Conclusion": "FILL IN",
    },
    {
        "Ablation": f"2: Feature Selection (best: {best_abl2_name})",
        "Hypothesis": "Removing low-evidence features reduces noise",
        "Controlled Change": f"Best phase: {best_abl2_name}",
        "CV F1 (mean +/- std)": best_abl2_metrics["CV F1"],
        "Test F1 (Yes)": f"{best_abl2_metrics['Test F1 (Yes)']:.4f}",
        "Test Recall (Yes)": f"{best_abl2_metrics['Test Recall (Yes)']:.4f}",
        "Test ROC-AUC": f"{best_abl2_metrics['Test ROC-AUC']:.4f}",
        "Conclusion": "FILL IN",
    },
    {
        "Ablation": "3: Class Weight",
        "Hypothesis": "Aggressive class weight improves recall/F1",
        "Controlled Change": f"class_weight={{0:1, 1:{int(best_abl3['K'])}}} (baseline was K=5)",
        "CV F1 (mean +/- std)": abl3_df.loc[abl3_df["Test F1 (Yes)"].idxmax(), "CV F1"],
        "Test F1 (Yes)": f"{best_abl3['Test F1 (Yes)']:.4f}",
        "Test Recall (Yes)": f"{best_abl3['Test Recall (Yes)']:.4f}",
        "Test ROC-AUC": f"{best_abl3['Test ROC-AUC']:.4f}",
        "Conclusion": "FILL IN",
    },
    {
        "Ablation": "4: F2 Threshold",
        "Hypothesis": "F2-optimal threshold suits medical screening better",
        "Controlled Change": f"Threshold: {baseline_threshold} -> {f2_best_row['Threshold']:.2f} (F2-optimal)",
        "CV F1 (mean +/- std)": baseline_metrics["CV F1"],
        "Test F1 (Yes)": f"{f2_best_row['F1']:.4f}",
        "Test Recall (Yes)": f"{f2_best_row['Recall']:.4f}",
        "Test ROC-AUC": f"{baseline_metrics['Test ROC-AUC']:.4f}",
        "Conclusion": "FILL IN",
    },
])

print("=" * 120)
print("  FORMAL ABLATION LOG (Part C Deliverable)")
print("=" * 120)
print(ablation_log.to_string(index=False))

ablation_log.to_csv("ablation_log.csv", index=False)
print("\nSaved to: ablation_log.csv")

  FORMAL ABLATION LOG (Part C Deliverable)
                                     Ablation                                           Hypothesis                                             Controlled Change CV F1 (mean +/- std) Test F1 (Yes) Test Recall (Yes) Test ROC-AUC          Conclusion
                                     Baseline                                        N/A (control)                                                          None    0.3250 +/- 0.0035        0.3166            0.4062       0.8389 Control measurement
                                1: Risk Score Domain-weighted composite feature improves detection Add MedicalRiskScore feature (T1=4, T2=2, T3=1, T4=0.5, T5=0)    0.3251 +/- 0.0036        0.3166            0.4062       0.8389             FILL IN
2: Feature Selection (best: Phase A: Drop T5)         Removing low-evidence features reduces noise                                  Best phase: Phase A: Drop T5    0.3190 +/- 0.0059        0.3095            0.3837   

### Interpreting the Ablation Log

After running, fill in each "FILL IN" conclusion with:
- **Positive:** metric improved -> adopt this change
- **Neutral:** metric unchanged within noise -> change is unnecessary
- **Negative:** metric degraded -> reject this change

The final champion configuration for Part D (Failure Analysis) and Part E (Decision Making) should incorporate whichever ablations showed positive results.